# Module 08 — Classes and Encapsulation

## Exercise 08.5 — Design a class properly

An Account class. Small enough to finish, large enough that every decision in
Module 08 shows up.
Everything here is a DESIGN exercise: the tests define the required behaviour,
but several choices are yours. Write down each choice and the reason.
Run:  python ex05_bank.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 2. The attribute-lookup ladder

**The single most useful diagram in Part 2.** When you write `obj.x`, Python:

```text
1. type(obj).__mro__  -- looking for a DATA DESCRIPTOR named x
                         (something with __get__ AND __set__ -- e.g. @property)
                         found? call its __get__ and STOP.
2. obj.__dict__['x']  -- the instance's own dictionary
                         found? return it and STOP.
3. type(obj).__mro__  -- the class and its bases, in MRO order
                         found? return it (binding it if it is a function)
4. type(obj).__getattr__('x')   -- last-resort hook, if defined
5. AttributeError
```


Two consequences that explain a great deal:

**Instance attributes shadow class attributes** (step 2 beats step 3) — but
**properties beat instance attributes** (step 1 beats step 2). That ordering is
what makes `@property` able to intercept an attribute that used to be plain
data.

**A method is found on the class, not the instance.** Every instance of a class
shares one function object; the binding happens at lookup time.

In [ ]:
class Dog:
    def speak(self): return "woof"

d = Dog()
Dog.speak            # <function Dog.speak>       -- a plain function
d.speak              # <bound method Dog.speak>   -- function + instance
d.speak()            # == Dog.speak(d)

That is all `self` is: the first parameter, filled in by the binding. Python
makes it explicit rather than implicit, which is why you can do this:

In [ ]:
Dog.speak(d)                      # call it unbound
handler = d.speak                 # store a bound method as a callback
list(map(str.upper, ["a", "b"]))  # use an unbound method as a function

---

## Concept 5. `@property`: why Python has no getters

In Java you write getters from the start because changing a public field to a
method later breaks every caller. **In Python it does not**, because
`@property` intercepts attribute access at the same syntax.

In [ ]:
class Circle:
    def __init__(self, radius: float) -> None:
        self.radius = radius      # start plain. No getter, no setter.

    @property
    def area(self) -> float:      # a computed, read-only attribute
        return 3.14159 * self.radius ** 2

c = Circle(2)
c.area                            # 12.56...   -- no parentheses
c.area = 5                        # AttributeError: property has no setter

Adding validation later, without changing any call site:

In [ ]:
class Circle:
    def __init__(self, radius: float) -> None:
        self.radius = radius      # this now goes through the setter

    @property
    def radius(self) -> float:
        return self._radius

    @radius.setter
    def radius(self, value: float) -> None:
        if value <= 0:
            raise ValueError(f"radius must be positive, got {value}")
        self._radius = value

Every existing `c.radius` and `c.radius = 5` keeps working, now validated. This
is why **you should not write a getter and setter until you need one.** Start
with a plain attribute; promote it to a property when there is a reason.

Two things to watch:

**Infinite recursion.** Inside the property, use `self._radius`, never
`self.radius` — the latter calls the property again.

**Cheapness.** A property looks like an attribute, so callers assume it is
cheap. A property that issues a database query will be called in a loop by
someone who had no way to know. If it is expensive, make it a method named
`compute_x()`, or cache it:

In [ ]:
from functools import cached_property

class Dataset:
    @cached_property
    def stats(self) -> dict[str, float]:      # computed once, then stored
        return expensive_analysis(self.rows)  # in the instance __dict__

`cached_property` works by writing the result into `self.__dict__`, so step 2 of
the lookup ladder finds it on every subsequent access and the descriptor never
runs again. (Which means it needs a `__dict__` — it does not work with
`__slots__`.)

---

## Concept 6. `@classmethod` and `@staticmethod`

In [ ]:
class Temperature:
    def __init__(self, kelvin: float) -> None:
        self.kelvin = kelvin

    @classmethod
    def from_celsius(cls, c: float) -> "Temperature":
        return cls(c + 273.15)             # cls, not Temperature

    @classmethod
    def from_fahrenheit(cls, f: float) -> "Temperature":
        return cls.from_celsius((f - 32) * 5 / 9)

    @staticmethod
    def is_valid_kelvin(value: float) -> bool:
        return value >= 0                   # no self, no cls

**`@classmethod` is how Python does named constructors.** A class can have only
one `__init__`, so alternative constructors become classmethods. Using `cls`
rather than the class name means subclasses get the right type back:

In [ ]:
class Kelvin(Temperature): ...
Kelvin.from_celsius(0)          # a Kelvin, not a Temperature

**`@staticmethod` is a function that lives in the class's namespace.** It gets
neither `self` nor `cls`. If it does not use either, ask whether it should be a
module-level function — often the honest answer is yes. It earns its place when
the grouping genuinely aids discovery, or when subclasses should be able to
override it.

---

## Concept 8. Encapsulation that actually works

Since `private` does not exist, encapsulation in Python is about **not handing
out mutable internals** — the Module 02 lesson, applied to design.

In [ ]:
class Playlist:
    def __init__(self, tracks: list[str]) -> None:
        self._tracks = list(tracks)          # copy IN

    @property
    def tracks(self) -> tuple[str, ...]:
        return tuple(self._tracks)           # immutable view OUT

    def add(self, track: str) -> None:
        self._tracks.append(track)

Without the copy on the way in, the caller keeps a handle on your internal list.
Without the conversion on the way out, anyone can mutate it. The underscore
documents intent; the copies enforce it.

The alternatives, each with a trade-off:

| Return | Cost | Caller can |
|---|---|---|
| `tuple(self._tracks)` | O(n) copy | read, index, not mutate |
| `list(self._tracks)` | O(n) copy | mutate their own copy |
| `iter(self._tracks)` | O(1) | iterate once; sees later mutations |
| `MappingProxyType(d)` | O(1) | read a dict; a live view, not a snapshot |
| `self._tracks` | O(1) | **everything.** Not encapsulation. |

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: A class body is executable code
- Section 2: The attribute-lookup ladder
- Section 3: Class attributes versus instance attributes
- Section 4: There is no `private`
- Section 5: `@property`: why Python has no getters
- Section 6: `@classmethod` and `@staticmethod`
- Section 7: `__slots__`
- Section 8: Encapsulation that actually works

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

from datetime import date, datetime
from decimal import Decimal
from typing import Any

---

## `InsufficientFunds`

Raised when a withdrawal would exceed the available balance.

In [ ]:
class InsufficientFunds(Exception):
    """Raised when a withdrawal would exceed the available balance."""

---

## `Account`

A bank account.

In [ ]:
class Account:
    """A bank account.

    TODO 1  __init__(owner, opening_balance="0.00", overdraft_limit="0.00")
            Validate: owner non-empty; balances parse as Decimal; overdraft
            limit not negative. Store money as Decimal, never float (Module 03).

    TODO 2  `balance` must be a READ-ONLY property. There must be no way to
            write a balance directly -- it changes only through deposit,
            withdraw and transfer. That is the entire point of the class; if a
            caller can write self.balance, the class is a namespace, not an
            invariant.

    TODO 3  `available` -- a computed property: balance + overdraft_limit.

    TODO 4  deposit(amount) and withdraw(amount).
            Both reject zero and negative amounts (a negative deposit is a
            withdrawal that skips the overdraft check -- a real class of bug).
            withdraw raises InsufficientFunds when it would exceed `available`,
            and the exception must carry the requested and available amounts.

    TODO 5  transfer(other, amount).
            Think about failure: if the deposit into `other` raised, must the
            withdrawal be undone? Write down your answer and implement it.
            (There is no single right answer without a transaction manager --
            but there IS a wrong answer, which is not thinking about it.)

    TODO 6  A classmethod `open_joint(owner_a, owner_b, **kwargs)` returning a
            new account with a combined owner name. Use cls, not Account, and
            say why.

    TODO 7  `statement()` returning the transaction history WITHOUT letting the
            caller modify it. Pick one of the strategies from the README's table
            and justify it.

    TODO 8  __repr__ that is useful in a debugger and does NOT expose anything
            that should not appear in a log. Think about what an account
            number would mean here.

    TODO 9  Decide: __slots__ or not? Justify with reference to how many
            Account objects a real system holds at once.
    """

---

## `verify`

_verify_

In [ ]:
def verify() -> None:
    a = Account("Ada", "100.00")
    assert a.balance == Decimal("100.00")
    assert a.available == Decimal("100.00")

    a.deposit("50.00")
    assert a.balance == Decimal("150.00")

    a.withdraw("30.00")
    assert a.balance == Decimal("120.00")

    try:
        a.balance = Decimal("1000000")     # type: ignore[misc]
    except AttributeError:
        pass
    else:
        raise AssertionError("balance must be read-only")

    for bad in ["0.00", "-5.00"]:
        for method in (a.deposit, a.withdraw):
            try:
                method(bad)
            except ValueError:
                pass
            else:
                raise AssertionError(f"{method.__name__}({bad}) should raise")

    try:
        a.withdraw("999.00")
    except InsufficientFunds as exc:
        assert exc.requested == Decimal("999.00")     # type: ignore[attr-defined]
        assert exc.available == Decimal("120.00")     # type: ignore[attr-defined]
    else:
        raise AssertionError("overdraw must raise InsufficientFunds")

    o = Account("Bo", "10.00", overdraft_limit="100.00")
    assert o.available == Decimal("110.00")
    o.withdraw("60.00")
    assert o.balance == Decimal("-50.00")

    b = Account("Cy")
    a.transfer(b, "20.00")
    assert a.balance == Decimal("100.00")
    assert b.balance == Decimal("20.00")

    joint = Account.open_joint("Ada", "Bo", opening_balance="500.00")
    assert isinstance(joint, Account)
    assert "Ada" in joint.owner and "Bo" in joint.owner

    stmt = a.statement()
    before = len(list(stmt))
    try:
        stmt.append("forged entry")       # type: ignore[attr-defined]
    except AttributeError:
        pass
    assert len(list(a.statement())) == before, "statement must not be mutable"

    assert "Ada" in repr(a)
    print("all account checks passed")
    print(repr(a))
    for line in a.statement():
        print(" ", line)

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    verify()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.